# Data Engineering: Ingest, Store, Model and Orchestrate

Notebook for **Chapter 2** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch02.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 2.1 From files to tables: what engineering adds

A file is data at rest in one shape. A table in a database is data with rules: every column has a type, a key makes rows unique, a constraint can reject nonsense, an index makes searches fast, and a transaction guarantees that a load either fully happens or not at all. Engineering is the work of moving from the first to the second, again and again, without surprises.

## 2.2 Designing the schema

The process: (1) list the *entities* (orders, customers, products); (2) pick a **primary key** for each (the column that identifies a row); (3) put attributes with the entity they describe; (4) connect entities with **foreign keys**; (5) choose types. This arrangement, a central fact table (orders) surrounded by dimension tables (customers, products), is called a star schema, and it is what Power BI's model view and most warehouses expect.

In [ ]:
DATA = "data/"
import sqlite3, pandas as pd

DDL = """
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, signup_date TEXT NOT NULL, segment TEXT, age REAL);
CREATE TABLE products  (product_id INTEGER PRIMARY KEY, product TEXT UNIQUE NOT NULL, category TEXT, list_price REAL, unit_cost REAL, description TEXT);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    order_date TEXT NOT NULL, order_hour INTEGER, city TEXT, channel TEXT, category TEXT,
    product TEXT REFERENCES products(product), quantity INTEGER CHECK (quantity > 0),
    unit_price REAL, discount REAL CHECK (discount BETWEEN 0 AND 1), status TEXT,
    promised_minutes REAL, delivery_minutes REAL);
"""
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript(DDL)
for t in ["customers", "products", "orders"]:
    df = pd.read_csv(DATA + t + ".csv")
    if t == "orders":
        df = df.drop_duplicates()
    df.to_sql(t, con, if_exists="append", index=False)
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

customers 4000
products 32
orders 20000


## 2.3 Loading: full, incremental and idempotent

A **full load** replaces the whole table. It is simple and safe for small data, but wasteful when tomorrow's file adds 60 rows to 20,000. An **incremental load** adds only what is new, found with a **watermark**: the largest `order_date` already loaded. The danger is the word "only": if the job crashes and reruns, or the same file arrives twice, a naive append doubles the rows. The cure is an **upsert**: insert new keys, update existing ones.

In [ ]:
DATA = "data/"
import sqlite3, pandas as pd

df = pd.read_csv(DATA + "orders.csv").drop_duplicates()
cols = list(df.columns)
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INT, order_date TEXT, order_hour INT, city TEXT, channel TEXT, category TEXT, product TEXT, quantity INT, unit_price REAL, discount REAL, status TEXT, promised_minutes REAL, delivery_minutes REAL)")

def upsert(con, frame):
    sets = ", ".join(f"{c}=excluded.{c}" for c in cols if c != "order_id")
    sql = f"INSERT INTO orders ({','.join(cols)}) VALUES ({','.join('?'*len(cols))}) ON CONFLICT(order_id) DO UPDATE SET {sets}"
    rows = [tuple(None if pd.isna(v) else v for v in r) for r in frame.itertuples(index=False, name=None)]
    con.executemany(sql, rows)
    con.commit()

first = df[df.order_date < "2024-07-01"]
upsert(con, first)
print("after first load :", con.execute("SELECT COUNT(*), MAX(order_date) FROM orders").fetchone())

try:
    con.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)", [tuple(None if pd.isna(v) else v for v in r) for r in df.itertuples(index=False, name=None)])
except sqlite3.IntegrityError as e:
    print("naive append     : rejected ->", e)
    con.rollback()

upsert(con, df)
upsert(con, df)   # run it twice on purpose
print("after 2 upserts  :", con.execute("SELECT COUNT(*), MAX(order_date) FROM orders").fetchone(), "| distinct ids in file:", df.order_id.nunique())

after first load : (9457, '2024-06-30')
naive append     : rejected -> UNIQUE constraint failed: orders.order_id
after 2 upserts  : (20000, '2024-12-31') | distinct ids in file: 20000


## 2.4 Indexes and the shape of a query

Without an index, SQLite reads every row to find a customer's orders. With an index on `customer_id` it jumps straight to them, like a book index versus reading every page. The tool that shows the difference is `EXPLAIN QUERY PLAN`, and a simple timer shows the effect on our 20,000 rows.

In [ ]:
DATA = "data/"
import sqlite3, time

con = sqlite3.connect(":memory:")
sqlite3.connect(DATA + "shopkart.db").backup(con)   # work on a copy, leave the file untouched
q = "SELECT COUNT(*), SUM(quantity) FROM orders WHERE customer_id = 1500"
print(con.execute("EXPLAIN QUERY PLAN " + q).fetchall()[0][3])

def timeit(n=2000):
    t = time.perf_counter()
    for _ in range(n):
        con.execute(q).fetchone()
    return (time.perf_counter() - t) / n * 1e6

before = timeit()
con.execute("CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id)")
print(con.execute("EXPLAIN QUERY PLAN " + q).fetchall()[0][3])
after = timeit()
con.execute("DROP INDEX idx_orders_customer")
print("index made the lookup faster:", before > 3 * after)

SCAN orders
SEARCH orders USING INDEX idx_orders_customer (customer_id=?)
index made the lookup faster: True


## 2.5 Data quality checks that stop bad data

A quality check is a small test with a name, a rule and a severity. Run the checks after ingest and before anything downstream reads the table. Six families cover most needs.

In [ ]:
DATA = "data/"
import pandas as pd

def run_checks(o, c):
    checks = [
        ("order_date not null",      o.order_date.isna().sum(), 0),
        ("order_id unique",          o.order_id.duplicated().sum(), 0),
        ("status in allowed set",    (~o.status.isin(["delivered", "returned", "cancelled"])).sum(), 0),
        ("discount within 0..1",     (~o.discount.between(0, 1)).sum(), 0),
        ("quantity 1..20",           (~o.quantity.between(1, 20)).sum(), 0),
        ("customer exists",          (~o.customer_id.isin(c.customer_id)).sum(), 0),
        ("city missing (<1%)",       o.city.isna().sum(), int(0.01 * len(o))),
        ("delivery time 5..300 min", (~o.delivery_minutes.dropna().between(5, 300)).sum(), 0),
    ]
    return pd.DataFrame(checks, columns=["check", "offenders", "tolerance"]).assign(
        result=lambda d: (d.offenders <= d.tolerance).map({True: "pass", False: "FAIL"}))

o = pd.read_csv(DATA + "orders.csv")
c = pd.read_csv(DATA + "customers.csv")
print(run_checks(o, c).to_string(index=False))

check  offenders  tolerance result
     order_date not null          0          0   pass
         order_id unique        150          0   FAIL
   status in allowed set          0          0   pass
    discount within 0..1          0          0   pass
          quantity 1..20          5          0   FAIL
         customer exists          0          0   pass
      city missing (<1%)        120        201   pass
delivery time 5..300 min         20          0   FAIL


## 2.6 A small pipeline you can schedule

Now we assemble the pieces into one script with five steps: *extract* (read), *validate* (checks on raw), *transform* (clean rules from Chapter 3), *load* (upsert into a warehouse file), and *publish* (build a mart table). Each step is a function, each logs one line, and the whole is idempotent.

In [ ]:
DATA = "data/"
import sys, sqlite3, tempfile, os
import pandas as pd
sys.path.insert(0, DATA)
import prep

def extract():
    o, c, p = prep.load_raw(DATA)
    return o, c, p

def validate(o, c):
    bad_keys = (~o.customer_id.isin(c.customer_id)).sum()
    assert bad_keys == 0, f"{bad_keys} orders with unknown customers"
    assert o.order_date.notna().all(), "missing order dates"
    return {"rows_in": len(o)}

def transform():
    return prep.load_clean(DATA)

def load(wh, o, c, p):
    for name, df in (("orders", o), ("customers", c), ("products", p)):
        d = df.copy()
        for col in d.columns:
            if str(d[col].dtype).startswith("datetime"):
                d[col] = d[col].dt.strftime("%Y-%m-%d")
        d.to_sql(name, wh, if_exists="replace", index=False)   # full reload per run, idempotent

def publish(wh):
    wh.execute("DROP TABLE IF EXISTS daily_city_sales")
    wh.execute("""CREATE TABLE daily_city_sales AS
                  SELECT order_date, city, COUNT(*) AS orders, ROUND(SUM(revenue), 2) AS revenue
                  FROM orders GROUP BY order_date, city""")

def run(wh):
    o, c, p = extract()
    stats = validate(o, c)
    o2, c2, p2 = transform()
    load(wh, o2, c2, p2)
    publish(wh)
    stats["rows_out"] = wh.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
    stats["mart_rows"] = wh.execute("SELECT COUNT(*) FROM daily_city_sales").fetchone()[0]
    return stats

path = os.path.join(tempfile.mkdtemp(), "warehouse.db")
wh = sqlite3.connect(path)
print("run 1:", run(wh))
print("run 2:", run(wh))
print(wh.execute("SELECT * FROM daily_city_sales ORDER BY revenue DESC LIMIT 3").fetchall())

run 1: {'rows_in': 20150, 'rows_out': 19995, 'mart_rows': 1931}
run 2: {'rows_in': 20150, 'rows_out': 19995, 'mart_rows': 1931}
[('2024-11-21', 'Bengaluru', 33, 41450.0), ('2024-07-06', 'Mumbai', 26, 40218.0), ('2024-11-29', 'Mumbai', 28, 38678.75)]


## 2.7 File formats and where data lives

CSV is readable and universal, but it has no types, no compression and no schema. Analytical systems prefer **columnar** formats such as Parquet: values of one column are stored together, so a query that needs two of fourteen columns reads far less, and types survive the round trip. Our file sizes make the point on a small scale.

In [ ]:
DATA = "data/"
import os, sqlite3, pandas as pd
for f in ["orders.csv", "customers.csv", "products.csv", "shopkart.db"]:
    print(f"{f:15s}{os.path.getsize(DATA + f)/1024:9.0f} KB")
df = pd.read_csv(DATA + "orders.csv")
print("in memory as read     :", round(df.memory_usage(deep=True).sum() / 1024), "KB")
small = df.assign(city=df.city.astype("category"), channel=df.channel.astype("category"), category=df.category.astype("category"),
                  product=df["product"].astype("category"), status=df.status.astype("category"),
                  quantity=df.quantity.astype("int16"), order_hour=df.order_hour.astype("int8"))
print("after category dtypes :", round(small.memory_usage(deep=True).sum() / 1024), "KB")

orders.csv          1705 KB
customers.csv        113 KB
products.csv           3 KB
shopkart.db         1920 KB
in memory as read     : 7916 KB
after category dtypes : 2266 KB


## 2.8 Choosing the right tool for engineering tasks

## Chapter recap

## Practice set

Problem 2.9.1 Name the three layers of a simple warehouse and say which one you never overwrite.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch02.html